# 🗺️ 01b — Pipeline géographie & population

Construit `dim_geo_pop.parquet`, une ligne par département. Lancer
`00_config_commun.ipynb` avant. Fusionne 3 sources indépendantes, variables
structurelles considérées stables sur 2020-2025 (pas de dimension temporelle).

Fichiers dans `data/raw/geo_pop/` :

| Fichier | Source | | Période |
|---|---|---|---|
| `insee_demographie.csv` | CNAM, Cartographie des pathologies | https://www.data.gouv.fr/datasets/pathologies-effectif-de-patients-par-pathologie-sexe-classe-dage-et-territoire-departement-region | 2015-2024, annuel |
| `insee_urbanisation.xlsx` | INSEE, population par taille d'unité urbaine | https://www.insee.fr/fr/statistiques/5039853 | 2017 (zonage 2020, basé sur le recensement 2017) |
| `ameli_acces_soins_YYYY.xls` | Ameli, densité des professionnels de santé | https://www.assurance-maladie.ameli.fr/etudes-et-donnees/densite-professionnels-sante-liberaux-departement | 1 fichier/an, 2020-2024 |

Colonnes produites :

| Colonne | Source | | |
|---|---|---|---|
| `pop_totale` | INSEE démographie | `Npop`, `cla_age_5 == "tsage"` | dernière année dispo. Ne pas sommer `Npop` sur tous les `cla_age_5`, "tsage" est déjà l'agrégat tous âges (sinon population comptée deux fois) |
| `part_seniors` | INSEE démographie | `Npop` 65+ / `pop_totale` | |
| `part_jeunes` | INSEE démographie | `Npop` <15 ans / `pop_totale` | |
| `prev_resp_chronique` | INSEE démographie | `prev`, `patho_niv1 == "Maladies respiratoires chroniques (hors mucoviscidose)"`, `cla_age_5 == "tsage"` | proxy de vulnérabilité respiratoire, pas de catégorie équivalente pour allergie/bronchiolite dans la nomenclature CNAM, ne remplace pas les `taux_urgences_*` |
| `tx_urbain` | INSEE urbanisation | part en unité urbaine (%) | figé à 2017, pas de version plus récente à l'échelle département |
| `densite_med_gen` | Ameli | feuille "Généralistes et MEP" | moyenne 2020-2024, valeur fixe par département |
| `densite_spe` | Ameli | feuille "Spécialistes" | idem |

La densité médicale sert de contrôle : dans un département où l'accès aux
généralistes est faible, les patients vont mécaniquement plus aux urgences
pour des motifs qui auraient pu être traités en ville. Ça aide le modèle à
distinguer "plus d'urgences parce que plus de maladie" de "plus d'urgences
parce que moins d'accès aux soins de ville".

In [1]:
# Préambule : on se place dans le répertoire racine du projet et on ajoute le répertoire courant au PYTHONPATH pour pouvoir importer src/config.py

# pour recharger automatiquement les modules modifiés （src config surtout） sans redémarrer le kernel
%load_ext autoreload 
%autoreload 2

import os
import sys
from pathlib import Path
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
from src.config import RAW_DIR, TABLES_DIR, ANNEE_DEBUT, ANNEE_FIN, DEPTS,  DEPT_NOM_TO_CODE
from src.validation import valider_dim_table

print(
    f"Config chargée depuis src/config.py : {len(DEPTS)} départements | {ANNEE_DEBUT}–{ANNEE_FIN}")
print(f"RAW_DIR    = {RAW_DIR}")
print(f"TABLES_DIR = {TABLES_DIR}")

Config chargée depuis src/config.py : 96 départements | 2020–2025
RAW_DIR    = /Users/siranh/Documents/Data Scientest/projet_liora/data/raw
TABLES_DIR = /Users/siranh/Documents/Data Scientest/projet_liora/data/processed


In [2]:
# Check les noms des variables dans les bases de données INSEE et Ameli

# INSEE démographie
df = pd.read_csv("data/raw/geo_pop/insee_demographie.csv", sep=";", nrows=3)
print("=== INSEE démographie ===")
print(df.columns.tolist())
display(df.head(2))

# INSEE urbanisation
df2 = pd.read_excel(
    "data/raw/geo_pop/insee_urbanisation.xlsx",
    sheet_name=1,
    header=None,
    skiprows=2  # sauter les 2 premières lignes de métadonnées
)
print("\n=== INSEE urbanisation ===")
display(df2.head(10))

# Ameli (une année) : plusieurs sheets 
df3 = pd.read_excel("data/raw/geo_pop/ameli_acces_soins_2022.xls", nrows=3)
print("\n=== Ameli 2022 ===")
print(df3.columns.tolist())
display(df3.head(2)) 

=== INSEE démographie ===
['annee', 'patho_niv1', 'patho_niv2', 'patho_niv3', 'top', 'cla_age_5', 'sexe', 'region', 'dept', 'Ntop', 'Npop', 'prev', 'Niveau prioritaire', 'libelle_classe_age', 'libelle_sexe', 'tri']


,annee,patho_niv1,patho_niv2,patho_niv3,top,cla_age_5,sexe,region,dept,Ntop,Npop,prev,Niveau prioritaire,libelle_classe_age,libelle_sexe,tri
0,2020,Maladies psychiatriques,Troubles psychotiques,Troubles psychotiques,PSY_PSC_IND,50-54,2,1,971,280,17040,1.643,"2,3",de 50 à 54 ans,femmes,48.0
1,2020,Maladies psychiatriques,Troubles psychotiques,Troubles psychotiques,PSY_PSC_IND,50-54,2,3,999,50,6090,0.771,"2,3",de 50 à 54 ans,femmes,48.0



=== INSEE urbanisation ===


,0,1,2
0,01,Ain,67.0
1,02,Aisne,53.2
2,03,Allier,58.3
3,04,Alpes-de-Haute-Provence,61.9
4,05,Hautes-Alpes,59.5
5,06,Alpes-Maritimes,95.9
6,07,Ardèche,63.3
7,08,Ardennes,57.3
8,09,Ariège,55.5
9,10,Aube,61.3



=== Ameli 2022 ===
['Année : ', 2022]


,Année :,2022
0,NaN,NaN
1,Sources :,les données de ce fichier sont issues de Amos ...


In [3]:
def build_dim_geo_pop() -> pd.DataFrame:
    """
    Construit la table dim_geo_pop en fusionnant 3 sources :
      1. INSEE démographie : population totale, structure par âge
      2. INSEE urbanisation : taux d'urbanisation par département
      3. Ameli (2020-2024)  : densité de professionnels de santé

    CLÉ PRIMAIRE : dept (une ligne par département)

    VARIABLES PRODUITES :
    ────────────────────
    dept              → code département (ex: "75")
    pop_totale        → population totale
    part_seniors      → part des 65 ans et plus
    part_jeunes       → part des moins de 15 ans
    tx_urbain         → taux de population en zone urbaine (%)
    densite_med_gen   → densité médecins généralistes /100k hab
    densite_spe       → densité médecins spécialistes /100k hab
    prev_resp_chronique → prévalence des maladies respiratoires chroniques
                          (hors mucoviscidose) en population, dernière année
                          disponible (proxy de vulnérabilité respiratoire de
                          fond ; ne pas confondre avec taux_urgences_asthme,
                          qui mesure une part d'activité aux urgences)
    """

    geo_pop_dir = RAW_DIR / "geo_pop"

    # ══════════════════════════════════════════════════════════════════
    # SOURCE 1 : INSEE démographie
    # ══════════════════════════════════════════════════════════════════
    # Ce fichier contient les effectifs de patients par pathologie,
    # âge et département. On l'utilise pour extraire la structure
    # démographique (population par tranche d'âge).
    #
    # Colonnes utiles :
    #   annee      → année
    #   dept       → code département
    #   cla_age_5  → classe d'âge (00-04, 05-09, ..., 85+)
    #   Npop       → population de référence pour ce groupe
    #   sexe       → 9 = tous sexes confondus

    print("Chargement INSEE démographie...")
    df_demo = pd.read_csv(
        geo_pop_dir / "insee_demographie.csv",
        sep=";", encoding="utf-8", low_memory=False
    )
    print(f"  Brut : {len(df_demo):,} lignes")

    # Garder uniquement : tous sexes (sexe=9) + dernière année dispo
    df_demo = df_demo[df_demo["sexe"] == 9].copy()
    derniere_annee = df_demo["annee"].max()
    df_demo = df_demo[df_demo["annee"] == derniere_annee]
    print(
        f"  Après filtre (sexe=9, annee={derniere_annee}) : {len(df_demo):,} lignes")

    # Nettoyage département
    df_demo["dept"] = (
        df_demo["dept"].astype(str).str.strip().str.upper().str.zfill(2)
    )

    # Conversion numérique
    df_demo["Npop"] = pd.to_numeric(df_demo["Npop"], errors="coerce")
    df_demo["cla_age_5"] = df_demo["cla_age_5"].astype(str).str.strip()

    # Prévalence des maladies respiratoires chroniques
    df_resp = df_demo[
        (df_demo["patho_niv1"] == "Maladies respiratoires chroniques (hors mucoviscidose)")
        & (df_demo["cla_age_5"] == "tsage")
    ][["dept", "prev"]].rename(columns={"prev": "prev_resp_chronique"})
    df_resp["prev_resp_chronique"] = pd.to_numeric(df_resp["prev_resp_chronique"], errors="coerce")
    print(f"  ✅ Prévalence respiratoire chronique : {len(df_resp)} départements (année {derniere_annee})")

    # Garder uniquement "Total consommants tous régimes" pour éviter le double-comptage par pathologie
    df_demo = df_demo[
        df_demo["patho_niv1"] == "Total consommants tous régimes"
    ].copy()
    print(f"  Après filtre patho totale : {len(df_demo):,} lignes")

    # Population totale par département
    # cla_age_5 == "tsage" est DÉJÀ l'agrégat "tous âges" fourni par le
    # fichier. 
    df_pop_totale = (
        df_demo[df_demo["cla_age_5"] == "tsage"][["dept", "Npop"]]
        .rename(columns={"Npop": "pop_totale"})
    )

    # Seniors : 65 ans et plus
    df_seniors = (
        df_demo[df_demo["cla_age_5"].str[:2].isin(
            ["65", "70", "75", "80", "85", "90", "95"]
        )]
        .groupby("dept")["Npop"].sum()
        .reset_index()
        .rename(columns={"Npop": "nb_seniors"})
    )

    # Jeunes : moins de 15 ans
    df_jeunes = (
        df_demo[df_demo["cla_age_5"].str[:2].isin(["00", "05", "10"])]
        .groupby("dept")["Npop"].sum()
        .reset_index()
        .rename(columns={"Npop": "nb_jeunes"})
    )

    # Fusion et calcul des parts
    df_demographie = df_pop_totale.copy()
    df_demographie = df_demographie.merge(df_seniors, on="dept", how="left")
    df_demographie = df_demographie.merge(df_jeunes,  on="dept", how="left")
    df_demographie = df_demographie.merge(df_resp,    on="dept", how="left")
    df_demographie["part_seniors"] = (
        df_demographie["nb_seniors"] / df_demographie["pop_totale"]
    ).round(4)
    df_demographie["part_jeunes"] = (
        df_demographie["nb_jeunes"] / df_demographie["pop_totale"]
    ).round(4)
    df_demographie = df_demographie[["dept", "pop_totale", "part_seniors",
                                     "part_jeunes", "prev_resp_chronique"]]

    print(f"  ✅ Démographie : {len(df_demographie)} départements")

    # ══════════════════════════════════════════════════════════════════
    # SOURCE 2 : INSEE urbanisation
    # ══════════════════════════════════════════════════════════════════

    print("\nChargement INSEE urbanisation...")
    df_urban = pd.read_excel(
        geo_pop_dir / "insee_urbanisation.xlsx",
        sheet_name=1,      # 2ème feuille
        header=None,
        skiprows=2,        # sauter les 2 lignes de métadonnées
        usecols=[0, 2]     # colonne 0=dept, colonne 2=taux
        )
    
    df_urban.columns = ["dept", "tx_urbain"]
    df_urban["dept"] = (df_urban["dept"].astype(str).str.strip().str.zfill(2).str.upper())
    df_urban["tx_urbain"] = pd.to_numeric(df_urban["tx_urbain"], errors="coerce").round(1)
    df_urban = df_urban[df_urban["dept"].isin(DEPTS)].dropna()
    print(f"  ✅ Urbanisation : {len(df_urban)} départements")

    # ══════════════════════════════════════════════════════════════════
    # SOURCE 3 : Ameli — densité professionnels de santé
    # ══════════════════════════════════════════════════════════════════
    # 5 fichiers annuels (2020-2024), chacun avec plusieurs feuilles.
    # On prend la feuille "Généralistes et MEP" pour les médecins
    # généralistes et "Spécialistes" pour les spécialistes.
    #
    # Structure de chaque feuille :
    #   col0 → type de spécialité (ex: "01- Médecine générale")
    #   col1 → département (ex: "01- Ain")
    #   col2 → effectif
    #   col3 → population
    #   col4 → densité /100 000 hab  ← on prend ça

    print("\nChargement Ameli...")

    # Charger les 5 années et faire la moyenne
    annees_ameli = range(2020, 2025)
    dfs_gen = []
    dfs_spe = []

    for annee in annees_ameli:
        fpath = geo_pop_dir / f"ameli_acces_soins_{annee}.xls"
        if not fpath.exists():
            print(f"  ⚠️  Manquant : {fpath.name}")
            continue
        try:
            # ── Feuille "Généralistes et MEP" ──────────────────────────
            df_gen = pd.read_excel(fpath, sheet_name="Généralistes et MEP", header=0)
            df_gen.columns = ["type_ps", "dept_raw", "effectif",
                               "population", "densite"] + list(df_gen.columns[5:])
            df_gen = df_gen[df_gen["type_ps"].astype(str).str.contains(
                "Médecine générale", case=False, na=False
            )].copy()
            df_gen["dept"] = (
                df_gen["dept_raw"].astype(str)
                .str.extract(r"^(\d{2}|2[AB])", expand=False)
                .str.zfill(2)
            )
            df_gen["densite"] = pd.to_numeric(df_gen["densite"], errors="coerce")
            df_gen = df_gen[df_gen["dept"].isin(DEPTS)]
            df_gen = df_gen.groupby("dept")["densite"].mean().reset_index()
            df_gen["annee"] = annee
            dfs_gen.append(df_gen)

            # ── Feuille "Spécialistes" ──────────────────────────────────
            df_spe = pd.read_excel(fpath, sheet_name="Spécialistes", header=0)
            df_spe.columns = ["type_ps", "dept_raw", "effectif",
                               "population", "densite"] + list(df_spe.columns[5:])
            df_spe = df_spe[df_spe["type_ps"].astype(str).str.contains(
                "Médecine", case=False, na=False
            )].copy()
            df_spe["dept"] = (
                df_spe["dept_raw"].astype(str)
                .str.extract(r"^(\d{2}|2[AB])", expand=False)
                .str.zfill(2)
            )
            df_spe["densite"] = pd.to_numeric(df_spe["densite"], errors="coerce")
            df_spe = df_spe[df_spe["dept"].isin(DEPTS)]
            df_spe = df_spe.groupby("dept")["densite"].mean().reset_index()
            df_spe["annee"] = annee
            dfs_spe.append(df_spe)

            print(f"  ✅ Ameli {annee} chargé")
        except Exception as e:
            print(f"  ⚠️  Erreur {annee} : {e}")

    # Moyenne sur toutes les années disponibles
    if len(dfs_gen) > 0:
        df_ameli_gen = pd.concat(dfs_gen)
        df_ameli_gen = df_ameli_gen.groupby("dept")["densite"].mean().reset_index()
        df_ameli_gen["densite"] = df_ameli_gen["densite"].round(2)
        df_ameli_gen = df_ameli_gen.rename(columns={"densite": "densite_med_gen"})
    else:
        df_ameli_gen = pd.DataFrame(columns=["dept", "densite_med_gen"])

    if len(dfs_spe) > 0:
        df_ameli_spe = pd.concat(dfs_spe)
        df_ameli_spe = df_ameli_spe.groupby("dept")["densite"].mean().reset_index()
        df_ameli_spe["densite"] = df_ameli_spe["densite"].round(2)
        df_ameli_spe = df_ameli_spe.rename(columns={"densite": "densite_spe"})
    else:
        df_ameli_spe = pd.DataFrame(columns=["dept", "densite_spe"])

    print(f"  ✅ Ameli généralistes : {len(df_ameli_gen)} depts")
    print(f"  ✅ Ameli spécialistes : {len(df_ameli_spe)} depts")

    # ══════════════════════════════════════════════════════════════════
    # FUSION FINALE
    # ══════════════════════════════════════════════════════════════════
    # On part de la liste complète des départements et on joint
    # chaque source avec how="left" pour conserver tous les depts
    # même si une source est incomplète

    print("\nFusion des 3 sources...")
    dim_geo_pop = pd.DataFrame({"dept": DEPTS})
    dim_geo_pop = dim_geo_pop.merge(df_demographie, on="dept", how="left")
    dim_geo_pop = dim_geo_pop.merge(df_urban,       on="dept", how="left")
    dim_geo_pop = dim_geo_pop.merge(df_ameli_gen,   on="dept", how="left")
    dim_geo_pop = dim_geo_pop.merge(df_ameli_spe,   on="dept", how="left")

    # Rapport de couverture
    print("\n📊 Couverture :")
    for col in dim_geo_pop.columns[1:]:
        pct = dim_geo_pop[col].notna().mean() * 100
        barre = "█" * int(pct // 10) + "░" * (10 - int(pct // 10))
        statut = "✅" if pct > 80 else "⚠️ "
        print(f"  {statut} {col:<25} {barre} {pct:.0f}%")

    print(f"\n✅ dim_geo_pop : {dim_geo_pop.shape[0]} depts × "
          f"{dim_geo_pop.shape[1]} colonnes")
    return dim_geo_pop


# ══════════════════════════════════════════════════════════════════════════════
# EXÉCUTION
# ══════════════════════════════════════════════════════════════════════════════
dim_geo_pop = build_dim_geo_pop()

if not dim_geo_pop.empty:
    valider_dim_table(dim_geo_pop, "dim_geo_pop", cle=["dept"])
    dim_geo_pop.to_parquet(TABLES_DIR / "dim_geo_pop.parquet", index=False)
    print(f"\n Sauvegardé → data/processed/dim_geo_pop.parquet")
    display(dim_geo_pop.head(10))

Chargement INSEE démographie...
  Brut : 5,796,000 lignes
  Après filtre (sexe=9, annee=2024) : 199,080 lignes
  ✅ Prévalence respiratoire chronique : 120 départements (année 2024)
  Après filtre patho totale : 2,520 lignes
  ✅ Démographie : 462 départements

Chargement INSEE urbanisation...
  ✅ Urbanisation : 96 départements

Chargement Ameli...
  ✅ Ameli 2020 chargé
  ✅ Ameli 2021 chargé
  ✅ Ameli 2022 chargé
  ✅ Ameli 2023 chargé
  ✅ Ameli 2024 chargé
  ✅ Ameli généralistes : 96 depts
  ✅ Ameli spécialistes : 96 depts

Fusion des 3 sources...

📊 Couverture :
  ✅ pop_totale                ██████████ 100%
  ✅ part_seniors              ██████████ 100%
  ✅ part_jeunes               ██████████ 100%
  ✅ prev_resp_chronique       ██████████ 100%
  ✅ tx_urbain                 ██████████ 100%
  ✅ densite_med_gen           ██████████ 100%
  ✅ densite_spe               ██████████ 100%

✅ dim_geo_pop : 96 depts × 8 colonnes
── Validation de dim_geo_pop ──
  ✅ Tous les codes dept sont valides (9

,dept,pop_totale,part_seniors,part_jeunes,prev_resp_chronique,tx_urbain,densite_med_gen,densite_spe
0,01,635420,0.2055,0.1798,5.586,67.0,5.88,0.11
1,02,511150,0.2316,0.1684,7.617,53.2,6.48,0.16
2,03,326580,0.3011,0.1365,6.255,58.3,7.69,0.34
3,04,167890,0.2844,0.1403,6.442,61.9,9.92,0.43
4,05,146080,0.2731,0.1397,5.245,59.5,12.47,0.61
5,06,1163540,0.2527,0.1469,6.069,95.9,10.82,0.59
6,07,331690,0.2724,0.1444,5.770,63.3,7.25,0.43
7,08,256360,0.2465,0.1577,8.583,57.3,7.87,0.16
8,09,154080,0.2876,0.1336,7.396,55.5,8.73,0.28
9,10,296710,0.2460,0.1605,6.194,61.3,6.25,0.23
